In [1]:
#!pip install pytorch-mppi[tune]
import torch
from pytorch_mppi import MPPI
from pytorch_mppi import autotune

import os
import argparse
import importlib.util

import gym
from crowd_nav.policy.policy_factory import policy_factory
from crowd_sim.envs.utils.robot import Robot
from crowd_sim.envs.policy.orca import ORCA
from crowd_sim.envs.utils.state import JointState

/home/socnav/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cpu'

config_file = os.path.join('data/output', 'config.py')
spec = importlib.util.spec_from_file_location('config', config_file)
if spec is None:
    parser.error('Config file not found.')
config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config)

env_config = config.EnvConfig(False)
env = gym.make('CrowdSim-v0')
env.configure(env_config)

# if args.square:
#     env.test_scenario = 'square_crossing'
# if args.circle:
#     env.test_scenario = 'circle_crossing'
# if args.test_scenario is not None:
#     env.test_scenario = args.test_scenario

#env.test_scenario = args.scenario

policy_config = config.PolicyConfig(False)
env_config = config.EnvConfig(False)
policy = policy_factory['vecmppi'](env_config)
policy.configure(policy_config)

robot = Robot(env_config, 'robot')
env.set_robot(robot)
robot.time_step = env.time_step
robot.set_policy(policy)

policy.set_phase('test')
policy.set_device(device)

policy.set_env(env)
robot.print_info()

ob = env.reset('test')

MPC:  {'model_predictor': 'SGAN_MPPI', 'params': {'dt': 0.2, 'prediction_horizon': 1.5, 'history_length': 8, 'action': {'span': 360, 'n_actions': 30}, 'cost': {'sigma': {'h': 0.3, 's': 0.6, 'r': 0.9}, 'q': {'obs': 100.0, 'goal': 1.0, 'wind': 5.0, 'dev': 1.0}, 'discrete_cost_type': 'winding'}, 'predictor': {'path': '/home/socnav/arstr/RelationalGraphLearning/crowd_nav/data/output/zara2_12_model.pt', 'use_gpu': False, 'num_samples': 20, 'deviation_penalty': True, 'use_sgan_action': False, 'use_sgan_mode': False}, 'log_cost': True}}
INITIALIZING VECMPPI
CONFIG:  <config.EnvConfig object at 0x7f412405dd00>
REAL_WORLD
SEED:  1000
YES THIS IS CORRECT


In [3]:
#device = "cpu"
#dtype = torch.double

# create MPPI with some initial parameters
mppi = MPPI(robot.policy.dynamics, robot.policy.cost, 5,
            noise_sigma=torch.diag(torch.tensor([5., 5.], device=device)),
            num_samples=500,
            horizon=6, device=device,
            lambda_=1,
            u_min=torch.tensor([-1, -1], dtype=torch.double, device=device),
            u_max=torch.tensor([1, 1], dtype=torch.double, device=device),
            step_dependent_dynamics=True)
#mppi = robot.policy.create_mppi()

In [4]:
# use the same nominal trajectory to start with for all the evaluations for fairness
nominal_trajectory = mppi.U.clone()
# parameters for our sample evaluation function - lots of choices for the evaluation function
evaluate_running_cost = True
num_refinement_steps = 5
num_trajectories = 5

def evaluate():
    costs = []
    rollouts = []
    # we sample multiple trajectories for the same start to goal problem, but in your case you should consider
    # evaluating over a diverse dataset of trajectories
    for j in range(num_trajectories):
        ob = env.reset('test')
        mppi.U = nominal_trajectory.clone()
        # the nominal trajectory at the start will be different if the horizon's changed
        mppi.change_horizon(mppi.T)
        # usually MPPI will have its nominal trajectory warm-started from the previous iteration
        # for a fair test of tuning we will reset its nominal trajectory to the same random one each time
        # we manually warm it by refining it for some steps
        state = JointState(robot.get_full_state(), ob)
        robot.policy.update_trajectory(state)
        robot.policy.update_predictions()
        robot.policy.update_goal(torch.Tensor([robot.gx, robot.gy]))
        start = robot.policy.flatten_state(robot.get_full_state())
        for k in range(num_refinement_steps):
            mppi.command(start, shift_nominal_trajectory=False)

        rollout = mppi.get_rollouts(start)

        this_cost = 0
        rollout = rollout[0]
        # here we evaluate on the rollout MPPI cost of the resulting trajectories
        # alternative costs for tuning the parameters are possible, such as just considering terminal cost
        if evaluate_running_cost:
            for t in range(len(rollout) - 1):
                this_cost = this_cost + robot.policy.cost(rollout[t], mppi.U[t], t)

        rollouts.append(rollout)
        costs.append(this_cost)
    # can return None for rollouts if they do not need to be calculated
    return autotune.EvaluationResult(torch.stack(costs), torch.stack(rollouts))

In [5]:
# these are subclass of TunableParameter (specifically MPPIParameter) that we want to tune
params_to_tune = [autotune.SigmaParameter(mppi), autotune.HorizonParameter(mppi), autotune.LambdaParameter(mppi)]
# create a tuner with a CMA-ES optimizer
tuner = autotune.Autotune(params_to_tune, evaluate_fn=evaluate, optimizer=autotune.CMAESOpt(sigma=1.0))
# tune parameters for a number of iterations
iterations = 30
for i in range(iterations):
  # results of this optimization step are returned
  res = tuner.optimize_step()
  # we can render the rollouts in the environment
  env.draw_rollouts(res.rollouts)
# get best results and apply it to the controller
# (by default the controller will take on the latest tuned parameter, which may not be best)
res = tuner.get_best_result()
tuner.apply_parameters(res.param_values)
print(res.param_values)

(5_w,10)-aCMA-ES (mu_w=3.2,w_1=45%) in dimension 4 (seed=9651, Thu Apr 18 20:58:35 2024)
SEED:  1001
YES THIS IS CORRECT
ARR SHAPE:  (36, 5)
trajectory:  (1, 36, 5)
TRAJECTORY INPUT SHAPE:  torch.Size([1, 36, 2])
OBS TRAJ SHAPE:  torch.Size([1, 36, 2]) torch.Size([1, 36, 2])
GENERATOR OUTPUT:  torch.Size([20, 5, 36, 2])
PRED TRAJ FAKE:  torch.Size([1, 20, 6, 35, 4])
PREDICTIONS SHAPE:  torch.Size([1, 20, 6, 35, 4])
IN GOAL COST STATE:  torch.Size([500, 2]) tensor([[ 0.3748, -1.0000],
        [ 1.0000,  0.2269],
        [ 1.0000,  1.0000],
        [ 1.0000,  0.0901],
        [ 0.5786,  1.0000],
        [ 0.3598,  1.0000],
        [ 0.0395, -0.3629],
        [ 1.0000, -1.0000],
        [ 0.7303, -1.0000],
        [ 1.0000, -0.5971],
        [ 1.0000,  1.0000],
        [ 1.0000,  1.0000],
        [ 1.0000,  1.0000],
        [-0.1860,  1.0000],
        [ 1.0000,  1.0000],
        [ 1.0000, -1.0000],
        [ 1.0000,  1.0000],
        [ 0.8927, -0.0978],
        [ 0.4888, -1.0000],
       

RuntimeError: The expanded size of the tensor (5) must match the existing size (2) at non-singleton dimension 1.  Target sizes: [1, 5].  Tensor sizes: [2]